In [ ]:
!nvidia-smi
!pip -q install trl datasets transformers accelerate

In [ ]:
%%writefile benchmark_grpo_t4.py
#!/usr/bin/env python3
"""Baseline GRPO benchmark on a single Tesla T4.

Purpose: measure tokens/sec, VRAM peak, and reward curve for on-policy RL
(TRL GRPOTrainer, Qwen2.5-0.5B-Instruct, GSM8K verifiable rewards) with NO
t4-cuda kernels yet. This is the baseline the fused-kernel run has to beat.

Run on Colab/Kaggle T4:
    pip install trl datasets transformers accelerate
    python benchmarks/benchmark_grpo_t4.py --steps 100

Results land in results/grpo_baseline/ as JSON + training log.
"""

import argparse
import json
import os
import re
import time

from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer

ANSWER_RE = re.compile(r"####\s*(-?[0-9][0-9,\.]*)")


def extract_answer(text: str) -> str | None:
    """Pull the final numeric answer from a GSM8K completion (or gold)."""
    m = ANSWER_RE.search(text)
    if m:
        return m.group(1).replace(",", "").rstrip(".")
    # fallback: last number in the text (models rarely use #### unprompted)
    nums = re.findall(r"-?[0-9][0-9,\.]*", text)
    return nums[-1].replace(",", "").rstrip(".") if nums else None


def reward_correct(completions, answer, **kwargs):
    """+1 exact match, 0 otherwise. Verifiable, no judge needed."""
    rewards = []
    for comp, gold in zip(completions, answer):
        pred = extract_answer(comp)
        gold = gold.replace(",", "").rstrip(".")
        rewards.append(1.0 if pred == gold else 0.0)
    return rewards


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen2.5-0.5B-Instruct")
    ap.add_argument("--steps", type=int, default=100)
    ap.add_argument("--batch-size", type=int, default=8,
                    help="prompts per step")
    ap.add_argument("--num-generations", type=int, default=8,
                    help="completions per prompt (GRPO group size)")
    ap.add_argument("--max-completion-len", type=int, default=256)
    ap.add_argument("--out-dir", default="results/grpo_baseline")
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    ds = load_dataset("openai/gsm8k", "main", split="train")
    ds = ds.shuffle(seed=42)
    # strip the #### answer from the prompt, keep it in `answer` column for the reward fn
    ds = ds.map(
        lambda ex: {
            "prompt": [
                {"role": "user",
                 "content": ex["question"] +
                 "\nEnd your solution with: #### <answer>"}
            ],
            "answer": ex["answer"].split("####")[-1].strip(),
        },
        remove_columns=["question", "answer"],
    )

    cfg = GRPOConfig(
        output_dir=args.out_dir,
        per_device_train_batch_size=args.batch_size,
        num_generations=args.num_generations,
        max_completion_length=args.max_completion_len,
        max_prompt_length=384,
        learning_rate=1e-6,
        max_steps=args.steps,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        bf16=False,  # T4 is Turing: no bf16
        fp16=True,
        gradient_checkpointing=True,
    )

    trainer = GRPOTrainer(
        model=args.model,
        reward_funcs=reward_correct,
        args=cfg,
        train_dataset=ds,
    )

    t0 = time.time()
    trainer.train()
    wall = time.time() - t0

    # tokens/sec estimate: completion tokens actually generated per step
    gen_tokens_per_step = (args.batch_size * args.max_completion_len)
    total_gen_tokens = gen_tokens_per_step * args.steps

    metrics = {
        "model": args.model,
        "steps": args.steps,
        "batch_size": args.batch_size,
        "num_generations": args.num_generations,
        "wall_seconds": round(wall, 1),
        "approx_generated_tokens": total_gen_tokens,
        "approx_tokens_per_sec": round(total_gen_tokens / wall, 1),
        "train_runtime": trainer.state.log_history[-1].get("train_runtime"),
        "final_reward": trainer.state.log_history[-1].get("reward", None),
        "reward_history": [
            {"step": h.get("step"), "reward": h.get("reward")}
            for h in trainer.state.log_history if "reward" in h
        ],
    }
    out_path = os.path.join(args.out_dir, "metrics.json")
    with open(out_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))
    print(f"\nWrote {out_path}")
    print("NOTE: peak VRAM is visible in nvidia-smi / Colab GPU panel; "
          "add torch.cuda.max_memory_allocated() capture in the next pass.")


if __name__ == "__main__":
    main()


In [ ]:
!python benchmark_grpo_t4.py --steps 100

In [ ]:
!cat results/grpo_baseline/metrics.json